In [1]:
import os
os.environ["RMHD_PRECISION"] = "64"

import jax
import jax_rmhd as jr
import jax.numpy as jnp
import jax.numpy.fft as ft
import matplotlib.pyplot as plt
import jax_rmhd.snapshot_io as sn
import jax_rmhd.timestepping as ts
import jax_rmhd.physics as phys
jr.init_cluster()

#parameters
nx = 256
ny = 256
nz = 256
Lx = 2.0 * jnp.pi
Ly = 2.0 * jnp.pi
Lz = 2.0 * jnp.pi

t_snap = 0.1
t_end = 1.0

cfl_safety=1.0 #this is pretty aggressive
spatial_dimensions=3
snap_path="data/test_lsrk/"

#we will use hyperviscosity
visc=1e-9
res=1e-9
hyper=3

x = jnp.linspace(0, Lx, nx, endpoint=False)
y = jnp.linspace(0, Ly, ny, endpoint=False)
z = jnp.linspace(0, Lz, nz, endpoint=False)
x_grid = x.reshape(1,-1,1)
y_grid = y.reshape(1,1,-1)
z_grid = z.reshape(-1,1,1)


#initialize arrays
#setting to zero to just test core code
phi = jnp.zeros_like(x_grid) + jnp.zeros_like(y_grid) + jnp.zeros_like(z_grid)
psi = jnp.zeros_like(phi)

#fft
phik=ft.rfft2(phi)
psik=ft.rfft2(psi)

#set up orbax snapshot manager
mngr=jr.snapshot_manager_setup(snap_path=snap_path,nsnap=1000)

#prepare necessary objects for simulation
params=jr.Parameters(nx=nx,ny=ny,nz=nz,Lx=Lx,Ly=Ly,Lz=Lz,visc=visc,res=res,hyper=hyper,cfl_safety=cfl_safety,dims=spatial_dimensions)
shardings=jr.setup_sharding(params)
kgrid = jr.setup_kgrids(params)
state = jr.SimulationState(t=0.0,fields=jr.Fields(phik,psik))

nblock = jr.estimate_good_nblock(state,kgrid,params,t_snap,t_end,nblock_min=1)
print("nblock estimate: "+str(nblock))

rmhd-solver has initialized jax in 64bit precision.
Running in local mode. Total devices: 1


nblock estimate: 4


In [2]:
jr.simulate_scan(state,kgrid,params,nblock,t_snap,t_end,mngr,shardings)

Saving initial state as snapshot 0
0.09817477042468103
0.19634954084936207
Saving snapshot 1
0.29452431127404305
0.392699081698724
Saving snapshot 2
0.4908738521234049
0.589048622548086
Saving snapshot 3
0.6872233929727671
0.7853981633974483
Saving snapshot 4
0.8835729338221294
0.9817477042468106
Saving snapshot 5
1.0799224746714913
Saving final state as snapshot 6


'Ending simulation at t = 1.0799224746714913'

In [3]:
jr.simulate(state,kgrid,params,t_snap,t_end,mngr,shardings)

Saving initial state as snapshot 0
Saving snapshot 1 at t = 0.1227184630308513
Saving snapshot 2 at t = 0.2454369260617026
Saving snapshot 3 at t = 0.36815538909255374
Saving snapshot 4 at t = 0.4908738521234049
Saving snapshot 5 at t = 0.6135923151542563
Saving snapshot 6 at t = 0.7363107781851077
Saving snapshot 7 at t = 0.8590292412159591
Saving snapshot 8 at t = 0.9817477042468106
Saving snapshot 9 at t = 1.1044661672776614
Ending simulation at t = 1.1044661672776614


SimulationState(t=Array(1.10446617, dtype=float64), fields=Fields(phik=Array([[[0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
        ...,
        [0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j]],

       [[0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
        ...,
        [0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j]],

       [[0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 